## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "llama-3.3-70b-versatile"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
DB_NAME = "vector_db"
load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [12]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    temperature=0,
    model=MODEL,
    openai_api_key=GROQ_API_KEY,
    base_url=GROQ_BASE_URL
)

### These LangChain objects implement the method `invoke()`

In [11]:
retriever.invoke("Who is Avery?")

[]

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='Avery can refer to different individuals or entities, depending on the context. Here are a few possibilities:\n\n1. Avery (given name): Avery is a unisex given name that originated from the Old English words "aelf" (elf) and "ric" (ruler). It\'s commonly used in English-speaking countries and has become increasingly popular in recent years.\n2. Avery (surname): Avery is also a surname of English origin, derived from the Old English words "aelf" (elf) and "ric" (ruler). It\'s found in many countries, including the United States, the United Kingdom, and Canada.\n3. Avery Dennison: Avery Dennison is a multinational corporation that produces and distributes a wide range of products, including labels, packaging materials, and office supplies. The company was founded in 1935 and is headquartered in Glendale, California.\n4. Avery Bradley: Avery Bradley is an American professional basketball player who has played in the NBA for several teams, including the Boston Celtics an

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

"I don't have any information about Averi Lancaster in relation to Insurellm. Could you please provide more context or details about who Averi Lancaster is or how they might be connected to Insurellm? I'll do my best to help."

## What could possibly come next? 😂

In [10]:
gr.ChatInterface(answer_question).launch()

d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!